# schedule_model_run_coverage_gate_v1

目标：验证 QuantBench 全模型运行 ledger 从 dashboard 进入 schedule 通知，再进入 `src/api` paper intent 风控闸口。这里不重新训练模型，只复用已完成的 `research/quantbench_all_model_run_ledger_v1`，把链路证据写回本 card。

边界：`src/analysis` 继续负责 qlib/regime/window 的模型效果差异分析；skill 只用于抽象研究思路、提出 regime 切分和失败模式假设，不作为 dashboard、schedule、api 的功能入口。

In [1]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
import copy
import json
import os
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

CARD_REL = Path("research") / "schedule_model_run_coverage_gate_v1"
SOURCE_CARD_REL = Path("research") / "quantbench_all_model_run_ledger_v1"
CARD = ROOT / CARD_REL
SOURCE_CARD = ROOT / SOURCE_CARD_REL
LOCK_DIR = CARD / "locks"
LOCK_DIR_REL = CARD_REL / "locks"
DASHBOARD_EXPORT = CARD / "dashboard_export.json"
SCHEDULED_DASHBOARD_EXPORT = CARD / "scheduled_dashboard_export.json"
SCHEDULED_DASHBOARD_EXPORT_REL = CARD_REL / "scheduled_dashboard_export.json"

CARD.mkdir(parents=True, exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
assert SOURCE_CARD.exists(), SOURCE_CARD

print(f"ROOT={ROOT}")
print(f"SOURCE_CARD={SOURCE_CARD.relative_to(ROOT)}")

ROOT=/Users/rziiiii/dev/workspace
SOURCE_CARD=research/quantbench_all_model_run_ledger_v1


## 1. 读取真实 QuantBench ledger 并导出 dashboard payload

这一步确认 dashboard 层已经有完整的 `model_run_coverage` 和 `model_run_ledger`，并且公开 artifact 不依赖训练明细。

In [2]:
from src.dashboard import write_model_matrix_dashboard_export
from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once
from src.api import build_target_position_intent_from_dashboard

payload = write_model_matrix_dashboard_export(
    SOURCE_CARD,
    output_path=DASHBOARD_EXPORT,
    top_n=8,
)
coverage = payload["model_run_coverage"]
top_model = payload["top_models"][0]["model"]

assert coverage["schema_version"] == "model.run_ledger_summary.v1"
assert coverage["all_models_completed"] is True
assert coverage["expected_model_count"] == coverage["trained_model_count"] == coverage["backtested_model_count"]
assert payload["model_run_ledger"]

coverage_summary = {
    "top_model": top_model,
    "expected_model_count": coverage["expected_model_count"],
    "trained_model_count": coverage["trained_model_count"],
    "backtested_model_count": coverage["backtested_model_count"],
    "missing_models": coverage["missing_models"],
}
coverage_summary

{'top_model': 'sthan_mse',
 'expected_model_count': 32,
 'trained_model_count': 32,
 'backtested_model_count': 32,
 'missing_models': []}

## 2. schedule 传播覆盖率到详细通知

schedule 不重新计算模型，只把 dashboard payload 的覆盖率字段转成定时任务 payload 和通知文本。

In [3]:
events: list[str] = []

ledger_models = tuple(str(row["model"]) for row in payload["model_run_ledger"])


def source_card_runner(**kwargs):
    return {
        "status": "source_card_reused",
        "variant_count": len(ledger_models),
        "source_card": str(SOURCE_CARD.relative_to(ROOT)),
    }

config = ModelDashboardPipelineConfig(
    name="model_dashboard_refresh",
    card_dir=SOURCE_CARD_REL,
    dashboard_output_path=SCHEDULED_DASHBOARD_EXPORT_REL,
    start="2020-01-01",
    end="2026-06-09",
    train_end="2025-12-31",
    model_names=ledger_models,
    dashboard_top_n=8,
    notify_details=True,
)

schedule_result = run_model_dashboard_refresh_once(
    config,
    lock_dir=LOCK_DIR_REL,
    notify=events.append,
    matrix_runner=source_card_runner,
)

schedule_result_payload = {
    "name": schedule_result.name,
    "status": schedule_result.status,
    "detail": schedule_result.detail,
    "payload": schedule_result.payload,
}

assert schedule_result.status == "success"
assert schedule_result.payload["model_run_all_completed"] is True
assert schedule_result.payload["model_run_expected_model_count"] == coverage["expected_model_count"]
assert schedule_result.payload["model_run_backtested_model_count"] == coverage["backtested_model_count"]
assert "run_coverage=32/32 backtested=32/32 missing=0" in events[0]

(CARD / "schedule_result.json").write_text(
    json.dumps(schedule_result_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(CARD / "notification_events.json").write_text(
    json.dumps(events, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

events

['model_dashboard_refresh:success | top_model=sthan_mse | dashboard_manifest=analysis_manifest | attempts=1 | export=research/schedule_model_run_coverage_gate_v1/scheduled_dashboard_export.json | top_failure=return_stability_lag@trailing_1m::downtrend | top_review=weakness:block_or_rework@trailing_1m::downtrend | backtest=backtest.config.v1 folds=5 slippage_bps=1.0 | regime_gap=downtrend:ridge_ic->sthan_mse/spread=111.297 | promotion=promote_candidate:sthan_mse blocked=0 | run_coverage=32/32 backtested=32/32 missing=0']

## 3. API paper intent 使用运行记录作为闸口

完整 ledger 下，top model 允许进入 paper review；如果 top model 的运行记录不完整，即使 dashboard 排名靠前，也只能降到最小 paper 仓位。

In [4]:
complete_intent = build_target_position_intent_from_dashboard(
    payload,
    instrument="SOXL",
    min_position=0.1,
    max_position=0.75,
)

assert complete_intent.metadata["run_status"] == "complete"
assert complete_intent.metadata["model_run_all_completed"] is True
assert complete_intent.metadata["execution_gate"] == "paper_review"

(CARD / "paper_intent_complete.json").write_text(
    json.dumps(complete_intent.to_dict(), ensure_ascii=False, indent=2),
    encoding="utf-8",
)

{
    "target_position": complete_intent.target_position,
    "execution_gate": complete_intent.metadata["execution_gate"],
    "run_status": complete_intent.metadata["run_status"],
    "source_model": complete_intent.source_model,
}

{'target_position': 0.5728027886858583,
 'execution_gate': 'paper_review',
 'run_status': 'complete',
 'source_model': 'sthan_mse'}

In [5]:
incomplete_payload = copy.deepcopy(payload)
expected = int(incomplete_payload["model_run_coverage"]["expected_model_count"])
incomplete_payload["model_run_coverage"].update(
    {
        "backtested_model_count": expected - 1,
        "analysis_recorded_model_count": expected - 1,
        "manifested_model_count": expected - 1,
        "all_models_completed": False,
        "missing_models": [top_model],
    }
)

for row in incomplete_payload["model_run_ledger"]:
    if row.get("model") == top_model:
        row["backtested"] = False
        row["analysis_recorded"] = False
        row["manifested"] = False
        break
else:
    raise AssertionError(f"top model not found in ledger: {top_model}")

incomplete_intent = build_target_position_intent_from_dashboard(
    incomplete_payload,
    instrument="SOXL",
    min_position=0.1,
    max_position=0.75,
)

assert incomplete_intent.target_position == 0.1
assert incomplete_intent.metadata["execution_gate"] == "blocked_to_min_position"
assert incomplete_intent.metadata["run_status"] == "incomplete"
assert incomplete_intent.metadata["run_backtested"] is False
assert incomplete_intent.metadata["model_run_all_completed"] is False
assert incomplete_intent.metadata["model_run_missing_models"] == top_model

(CARD / "paper_intent_incomplete_gate.json").write_text(
    json.dumps(incomplete_intent.to_dict(), ensure_ascii=False, indent=2),
    encoding="utf-8",
)

{
    "target_position": incomplete_intent.target_position,
    "execution_gate": incomplete_intent.metadata["execution_gate"],
    "run_status": incomplete_intent.metadata["run_status"],
    "missing_models": incomplete_intent.metadata["model_run_missing_models"],
}

{'target_position': 0.1,
 'execution_gate': 'blocked_to_min_position',
 'run_status': 'incomplete',
 'missing_models': 'sthan_mse'}

In [6]:
notebook_summary = {
    "schema_version": "notebook.summary.v1",
    "artifact_id": "schedule_model_run_coverage_gate_v1",
    "source_card": str(SOURCE_CARD.relative_to(ROOT)),
    "dashboard_export": str(DASHBOARD_EXPORT.relative_to(ROOT)),
    "scheduled_dashboard_export": str(SCHEDULED_DASHBOARD_EXPORT.relative_to(ROOT)),
    "schedule_result": str((CARD / "schedule_result.json").relative_to(ROOT)),
    "notification_events": str((CARD / "notification_events.json").relative_to(ROOT)),
    "complete_intent": str((CARD / "paper_intent_complete.json").relative_to(ROOT)),
    "incomplete_intent": str((CARD / "paper_intent_incomplete_gate.json").relative_to(ROOT)),
    "top_model": top_model,
    "coverage_all_completed": bool(coverage["all_models_completed"]),
    "schedule_notification": events[0],
    "complete_execution_gate": complete_intent.metadata["execution_gate"],
    "incomplete_execution_gate": incomplete_intent.metadata["execution_gate"],
    "decision": "keep: schedule/api now consume model run ledger coverage as an execution guardrail",
}

(CARD / "notebook_summary.json").write_text(
    json.dumps(notebook_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

notebook_summary

{'schema_version': 'notebook.summary.v1',
 'artifact_id': 'schedule_model_run_coverage_gate_v1',
 'source_card': 'research/quantbench_all_model_run_ledger_v1',
 'dashboard_export': 'research/schedule_model_run_coverage_gate_v1/dashboard_export.json',
 'scheduled_dashboard_export': 'research/schedule_model_run_coverage_gate_v1/scheduled_dashboard_export.json',
 'schedule_result': 'research/schedule_model_run_coverage_gate_v1/schedule_result.json',
 'notification_events': 'research/schedule_model_run_coverage_gate_v1/notification_events.json',
 'complete_intent': 'research/schedule_model_run_coverage_gate_v1/paper_intent_complete.json',
 'incomplete_intent': 'research/schedule_model_run_coverage_gate_v1/paper_intent_incomplete_gate.json',
 'top_model': 'sthan_mse',
 'coverage_all_completed': True,
 'schedule_notification': 'model_dashboard_refresh:success | top_model=sthan_mse | dashboard_manifest=analysis_manifest | attempts=1 | export=research/schedule_model_run_coverage_gate_v1/schedu

## 4. 单次结论

本次验证完成：`model_run_coverage` 和 `model_run_ledger` 现在不只停留在 dashboard 展示层，也会进入 schedule 详细通知和 `src/api` paper intent metadata。完整 ledger 下 top model 进入 `paper_review`；当 top model 的训练/预测/回测/目标评估/晋级评估/分析记录/manifest 任一运行记录不完整时，API 会把目标仓位降到最小值。

对架构的含义：`src/analysis` 负责 qlib/regime/window 的稳定分析输出，dashboard 读取公开 manifest，schedule 传播摘要，api 只消费已发布的 dashboard/analysis/run-ledger 结果。skill 不承载项目功能，只用于生成分析假设和发散检查清单。